In [59]:
import pandas as pd
import sqlite3

print("--- Loading and cleaning source files ---")

departments = pd.read_csv('departments.csv')
jobs = pd.read_csv('jobs.csv')
employees = pd.read_csv('employees.csv')
salaries = pd.read_csv('salaries.csv')
performance = pd.read_csv('performance_reviews.csv')
attendance = pd.read_csv('attendance.csv')

employees['first_name'] = employees['first_name'].str.strip()
employees['last_name'] = employees['last_name'].str.strip()
employees['email'] = employees['email'].str.lower()

def clean_gender(g):
    if pd.isna(g):
        return 'Unknown'
    g_str = str(g).strip().lower()
    if g_str in ['m', 'male']:
        return 'Male'
    elif g_str in ['f', 'female']:
        return 'Female'
    return g_str

employees['gender'] = employees['gender'].apply(clean_gender)

employees['birth_date'] = pd.to_datetime(employees['birth_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['hire_date'] = pd.to_datetime(employees['hire_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_reason'] = employees['termination_reason'].fillna('Active')

initial_attendance_rows = len(attendance)
attendance = attendance.drop_duplicates()
print(f"Removed {initial_attendance_rows - len(attendance)} duplicate rows from attendance.")

conn = sqlite3.connect('hr_warehouse.db')

departments.to_sql('dim_department', conn, if_exists='replace', index=False)
jobs.to_sql('dim_job', conn, if_exists='replace', index=False)
employees.to_sql('dim_employee', conn, if_exists='replace', index=False)
salaries.to_sql('fact_salary', conn, if_exists='replace', index=False)
performance.to_sql('fact_performance', conn, if_exists='replace', index=False)
attendance.to_sql('fact_attendance', conn, if_exists='replace', index=False)

conn.close()
print("--- Database 'hr_warehouse.db' created and populated successfully! ---")

--- Loading and cleaning source files ---
Removed 290 duplicate rows from attendance.


/tmp/ipykernel_2332/3152376933.py:31: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')


--- Database 'hr_warehouse.db' created and populated successfully! ---


In [60]:
import sqlite3

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS fact_headcount_monthly;

CREATE TABLE fact_headcount_monthly AS
WITH RECURSIVE months(ym) AS (
    SELECT '2018-01'
    UNION ALL
    SELECT strftime('%Y-%m', ym || '-01', '+1 month')
    FROM months
    WHERE ym < '2026-06'
)
SELECT
    m.ym AS year_month,
    COUNT(e.emp_id) AS headcount
FROM months m
JOIN dim_employee e ON e.hire_date <= m.ym || '-28'
AND (e.termination_date IS NULL OR e.termination_date > m.ym || '-28')
GROUP BY m.ym;
""")

query_q2_updated = """
SELECT
    SUBSTR(year_month, 1, 4) AS year,
    ROUND(AVG(headcount), 2) AS avg_headcount
FROM fact_headcount_monthly
GROUP BY year
ORDER BY year;
"""

cursor.execute(query_q2_updated)
print("--- Updated Q2 Results using fact_headcount_monthly ---")
for row in cursor.fetchall():
    print(row)

conn.close()

--- Updated Q2 Results using fact_headcount_monthly ---
('2018', 29.75)
('2019', 79.42)
('2020', 101.25)
('2021', 128.25)
('2022', 160.0)
('2023', 180.33)
('2024', 198.83)
('2025', 224.75)
('2026', 230.0)


In [61]:
import sqlite3

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

print("="*60)
print("Q1: Headcount & Average Salary per Department")
print("="*60)
query_q1 = """
SELECT
    d.dept_name,
    COUNT(e.emp_id) as headcount,
    ROUND(AVG(s.salary), 2) as avg_monthly_salary
FROM dim_department d
JOIN dim_employee e ON d.dept_id = e.dept_id
JOIN fact_salary s ON e.emp_id = s.emp_id
WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30'
GROUP BY d.dept_name
ORDER BY avg_monthly_salary DESC;
"""
cursor.execute(query_q1)
for row in cursor.fetchall():
    print(f"Department: {row[0]:<20} | Headcount: {row[1]:<5} | Avg Salary: ${row[2]}")

print("\n" + "="*60)
print("Q2: Yearly Attrition (Leavers)")
print("="*60)
query_q2 = """
SELECT
    SUBSTR(termination_date, 1, 4) as term_year,
    COUNT(*) as leavers
FROM dim_employee
WHERE termination_date IS NOT NULL
GROUP BY term_year
ORDER BY term_year;
"""
cursor.execute(query_q2)
for row in cursor.fetchall():
    print(f"Year: {row[0]} | Leavers: {row[1]}")

conn.close()

Q1: Headcount & Average Salary per Department
Department: Engineering          | Headcount: 62    | Avg Salary: $3391.77
Department: Sales                | Headcount: 49    | Avg Salary: $2888.98
Department: Finance              | Headcount: 36    | Avg Salary: $2766.11
Department: Marketing            | Headcount: 25    | Avg Salary: $2707.6
Department: Human Resources      | Headcount: 30    | Avg Salary: $2472.67
Department: Customer Support     | Headcount: 25    | Avg Salary: $2074.4

Q2: Yearly Attrition (Leavers)
Year: 2018 | Leavers: 1
Year: 2019 | Leavers: 12
Year: 2020 | Leavers: 24
Year: 2021 | Leavers: 11
Year: 2022 | Leavers: 18
Year: 2023 | Leavers: 25
Year: 2024 | Leavers: 29
Year: 2025 | Leavers: 24
Year: 2026 | Leavers: 29


In [62]:
import pandas as pd
import sqlite3

print("--- Loading and cleaning source files ---")

departments = pd.read_csv('departments.csv')
jobs = pd.read_csv('jobs.csv')
employees = pd.read_csv('employees.csv')
salaries = pd.read_csv('salaries.csv')
performance = pd.read_csv('performance_reviews.csv')
attendance = pd.read_csv('attendance.csv')

employees['first_name'] = employees['first_name'].str.strip()
employees['last_name'] = employees['last_name'].str.strip()
employees['email'] = employees['email'].str.lower()

def clean_gender(g):
    if pd.isna(g):
        return 'Unknown'
    g_str = str(g).strip().lower()
    if g_str in ['m', 'male']:
        return 'Male'
    elif g_str in ['f', 'female']:
        return 'Female'
    return g_str

employees['gender'] = employees['gender'].apply(clean_gender)
employees['birth_date'] = pd.to_datetime(employees['birth_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['hire_date'] = pd.to_datetime(employees['hire_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_reason'] = employees['termination_reason'].fillna('Active')

attendance = attendance.drop_duplicates()
attendance['is_unplanned_absence'] = attendance['status'].apply(lambda x: 1 if x in ['Sick', 'Absent'] else 0)

conn = sqlite3.connect('hr_warehouse.db')
departments.to_sql('dim_department', conn, if_exists='replace', index=False)
jobs.to_sql('dim_job', conn, if_exists='replace', index=False)
employees.to_sql('dim_employee', conn, if_exists='replace', index=False)
salaries.to_sql('fact_salary', conn, if_exists='replace', index=False)
performance.to_sql('fact_performance', conn, if_exists='replace', index=False)
attendance.to_sql('fact_attendance', conn, if_exists='replace', index=False)
conn.commit()
conn.close()

print("--- Database updated successfully with all columns! ---")

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

queries = [
    """
    SELECT d.dept_name, e.termination_reason, COUNT(*) as cnt
    FROM dim_employee e
    JOIN dim_department d ON e.dept_id = d.dept_id
    WHERE e.termination_reason != 'Active'
    GROUP BY d.dept_name, e.termination_reason
    ORDER BY cnt DESC LIMIT 5;
    """,
    """
    SELECT j.job_title, e.gender, ROUND(AVG(s.salary), 2) as avg_sal
    FROM dim_employee e
    JOIN dim_job j ON e.job_id = j.job_id
    JOIN fact_salary s ON e.emp_id = s.emp_id
    WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30'
    GROUP BY j.job_title, e.gender;
    """,
    """
    SELECT d.dept_name, j.job_title, s.salary
    FROM dim_employee e
    JOIN dim_department d ON e.dept_id = d.dept_id
    JOIN dim_job j ON e.job_id = j.job_id
    JOIN fact_salary s ON e.emp_id = s.emp_id
    WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND j.job_title NOT LIKE '%Manager%'
    GROUP BY d.dept_name
    ORDER BY s.salary DESC;
    """,
    """
    SELECT p.rating, COUNT(*) as cnt
    FROM fact_performance p
    GROUP BY p.rating;
    """,
    """
    SELECT d.dept_name, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct
    FROM fact_attendance a
    JOIN dim_employee e ON a.emp_id = e.emp_id
    JOIN dim_department d ON e.dept_id = d.dept_id
    WHERE a.work_date LIKE '2025%'
    GROUP BY d.dept_name
    ORDER BY unplanned_pct DESC;
    """,
    """
    SELECT p.rating, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct
    FROM fact_attendance a
    JOIN fact_performance p ON a.emp_id = p.emp_id
    WHERE a.work_date LIKE '2025%' AND p.review_year = 2024
    GROUP BY p.rating
    ORDER BY p.rating;
    """,
    """
    SELECT COUNT(*) as overpaid_count
    FROM dim_employee e
    JOIN dim_job j ON e.job_id = j.job_id
    JOIN fact_salary s ON e.emp_id = s.emp_id
    WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND s.salary > j.max_salary;
    """,
    """
    SELECT termination_reason, ROUND(AVG((julianday(termination_date) - julianday(hire_date)) / 365.25), 1) as avg_tenure_years
    FROM dim_employee
    WHERE termination_date IS NOT NULL
    GROUP BY termination_reason;
    """
]

for i, q in enumerate(queries, 3):
    cursor.execute(q)
    print(f"--- Q{i} Results ---")
    for row in cursor.fetchall():
        print(row)

conn.close()

--- Loading and cleaning source files ---


/tmp/ipykernel_2332/2380704571.py:30: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')


--- Database updated successfully with all columns! ---
--- Q3 Results ---
('Customer Support', 'Better Offer', 24)
('Customer Support', 'Layoff', 14)
('Sales', 'Better Offer', 14)
('Sales', 'Layoff', 13)
('Customer Support', 'Relocation', 9)
--- Q4 Results ---
('Associate', 'Female', 2355.17)
('Associate', 'Male', 2402.35)
('Junior Associate', 'Female', 1704.62)
('Junior Associate', 'Male', 1697.05)
('Manager', 'Female', 6734.0)
('Manager', 'Male', 7275.71)
('Senior Associate', 'Female', 3480.45)
('Senior Associate', 'Male', 3611.11)
('Team Lead', 'Female', 4534.62)
('Team Lead', 'Male', 5201.43)
--- Q5 Results ---
('Engineering', 'Junior Associate', 1850.0)
('Marketing', 'Junior Associate', 1720.0)
('Sales', 'Junior Associate', 1690.0)
('Finance', 'Junior Associate', 1480.0)
('Human Resources', 'Junior Associate', 1410.0)
('Customer Support', 'Junior Associate', 1270.0)
--- Q6 Results ---
(1, 55)
(2, 182)
(3, 498)
(4, 309)
(5, 121)
--- Q7 Results ---
('Customer Support', 7.33)
('Sale

In [63]:
import sqlite3

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

queries = [
    ("Q1", "SELECT d.dept_name, COUNT(e.emp_id) as headcount, ROUND(AVG(s.salary), 2) as avg_salary FROM dim_department d JOIN dim_employee e ON d.dept_id = e.dept_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30' GROUP BY d.dept_name ORDER BY avg_salary DESC;"),
    ("Q2", "SELECT SUBSTR(termination_date, 1, 4) as year, COUNT(*) as leavers FROM dim_employee WHERE termination_date IS NOT NULL GROUP BY year ORDER BY year;"),
    ("Q3", "SELECT d.dept_name, e.termination_reason, COUNT(*) as cnt FROM dim_employee e JOIN dim_department d ON e.dept_id = d.dept_id WHERE e.termination_reason != 'Active' GROUP BY d.dept_name, e.termination_reason ORDER BY cnt DESC LIMIT 5;"),
    ("Q4", "SELECT j.job_title, e.gender, ROUND(AVG(s.salary), 2) as avg_sal FROM dim_employee e JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30' GROUP BY j.job_title, e.gender;"),
    ("Q5", "SELECT d.dept_name, j.job_title, s.salary FROM dim_employee e JOIN dim_department d ON e.dept_id = d.dept_id JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND j.job_title NOT LIKE '%Manager%' GROUP BY d.dept_name ORDER BY s.salary DESC;"),
    ("Q6", "SELECT p.rating, COUNT(*) as cnt FROM fact_performance p GROUP BY p.rating;"),
    ("Q7", "SELECT d.dept_name, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct FROM fact_attendance a JOIN dim_employee e ON a.emp_id = e.emp_id JOIN dim_department d ON e.dept_id = d.dept_id WHERE a.work_date LIKE '2025%' GROUP BY d.dept_name ORDER BY unplanned_pct DESC;"),
    ("Q8", "SELECT p.rating, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct FROM fact_attendance a JOIN fact_performance p ON a.emp_id = p.emp_id WHERE a.work_date LIKE '2025%' AND p.review_year = 2024 GROUP BY p.rating ORDER BY p.rating;"),
    ("Q9", "SELECT COUNT(*) as overpaid_count FROM dim_employee e JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND s.salary > j.max_salary;"),
    ("Q10", "SELECT termination_reason, ROUND(AVG((julianday(termination_date) - julianday(hire_date)) / 365.25), 1) as avg_tenure_years FROM dim_employee WHERE termination_date IS NOT NULL GROUP BY termination_reason;")
]

for name, q in queries:
    cursor.execute(q)
    print(f"--- {name} Results ---")
    for row in cursor.fetchall():
        print(row)

conn.close()

--- Q1 Results ---
('Engineering', 62, 3391.77)
('Sales', 49, 2888.98)
('Finance', 36, 2766.11)
('Marketing', 25, 2707.6)
('Human Resources', 30, 2472.67)
('Customer Support', 25, 2074.4)
--- Q2 Results ---
('2018', 1)
('2019', 12)
('2020', 24)
('2021', 11)
('2022', 18)
('2023', 25)
('2024', 29)
('2025', 24)
('2026', 29)
--- Q3 Results ---
('Customer Support', 'Better Offer', 24)
('Customer Support', 'Layoff', 14)
('Sales', 'Better Offer', 14)
('Sales', 'Layoff', 13)
('Customer Support', 'Relocation', 9)
--- Q4 Results ---
('Associate', 'Female', 2355.17)
('Associate', 'Male', 2402.35)
('Junior Associate', 'Female', 1704.62)
('Junior Associate', 'Male', 1697.05)
('Manager', 'Female', 6734.0)
('Manager', 'Male', 7275.71)
('Senior Associate', 'Female', 3480.45)
('Senior Associate', 'Male', 3611.11)
('Team Lead', 'Female', 4534.62)
('Team Lead', 'Male', 5201.43)
--- Q5 Results ---
('Engineering', 'Junior Associate', 1850.0)
('Marketing', 'Junior Associate', 1720.0)
('Sales', 'Junior Assoc

In [64]:
import pandas as pd
import sqlite3

print("--- Step 1: Extract, Clean, and Load (ETL) ---")

departments = pd.read_csv('departments.csv')
jobs = pd.read_csv('jobs.csv')
employees = pd.read_csv('employees.csv')
salaries = pd.read_csv('salaries.csv')
performance = pd.read_csv('performance_reviews.csv')
attendance = pd.read_csv('attendance.csv')

employees['first_name'] = employees['first_name'].str.strip()
employees['last_name'] = employees['last_name'].str.strip()
employees['email'] = employees['email'].str.lower()

def clean_gender(g):
    if pd.isna(g):
        return 'Unknown'
    g_str = str(g).strip().lower()
    if g_str in ['m', 'male']:
        return 'Male'
    elif g_str in ['f', 'female']:
        return 'Female'
    return g_str

employees['gender'] = employees['gender'].apply(clean_gender)
employees['birth_date'] = pd.to_datetime(employees['birth_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['hire_date'] = pd.to_datetime(employees['hire_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_reason'] = employees['termination_reason'].fillna('Active')

attendance = attendance.drop_duplicates()
attendance['is_unplanned_absence'] = attendance['status'].apply(lambda x: 1 if x in ['Sick', 'Absent'] else 0)

conn = sqlite3.connect('hr_warehouse.db')
departments.to_sql('dim_department', conn, if_exists='replace', index=False)
jobs.to_sql('dim_job', conn, if_exists='replace', index=False)
employees.to_sql('dim_employee', conn, if_exists='replace', index=False)
salaries.to_sql('fact_salary', conn, if_exists='replace', index=False)
performance.to_sql('fact_performance', conn, if_exists='replace', index=False)
attendance.to_sql('fact_attendance', conn, if_exists='replace', index=False)
conn.commit()
conn.close()

print("--- Step 2: Running All 10 Analytical Queries ---")

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

queries = [
    ("Q1: Headcount & Avg Salary per Dept", "SELECT d.dept_name, COUNT(e.emp_id) as headcount, ROUND(AVG(s.salary), 2) as avg_salary FROM dim_department d JOIN dim_employee e ON d.dept_id = e.dept_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30' GROUP BY d.dept_name ORDER BY avg_salary DESC;"),
    ("Q2: Yearly Attrition (Leavers)", "SELECT SUBSTR(termination_date, 1, 4) as year, COUNT(*) as leavers FROM dim_employee WHERE termination_date IS NOT NULL GROUP BY year ORDER BY year;"),
    ("Q3: Top Leaver Depts & Reasons", "SELECT d.dept_name, e.termination_reason, COUNT(*) as cnt FROM dim_employee e JOIN dim_department d ON e.dept_id = d.dept_id WHERE e.termination_reason != 'Active' GROUP BY d.dept_name, e.termination_reason ORDER BY cnt DESC LIMIT 5;"),
    ("Q4: Gender Pay Gap by Job Title", "SELECT j.job_title, e.gender, ROUND(AVG(s.salary), 2) as avg_sal FROM dim_employee e JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30' GROUP BY j.job_title, e.gender;"),
    ("Q5: Top-Paid Non-Manager per Dept", "SELECT d.dept_name, j.job_title, s.salary FROM dim_employee e JOIN dim_department d ON e.dept_id = d.dept_id JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND j.job_title NOT LIKE '%Manager%' GROUP BY d.dept_name ORDER BY s.salary DESC;"),
    ("Q6: Performance Rating Distribution", "SELECT p.rating, COUNT(*) as cnt FROM fact_performance p GROUP BY p.rating;"),
    ("Q7: Unplanned Absence 2025 by Dept (%)", "SELECT d.dept_name, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct FROM fact_attendance a JOIN dim_employee e ON a.emp_id = e.emp_id JOIN dim_department d ON e.dept_id = d.dept_id WHERE a.work_date LIKE '2025%' GROUP BY d.dept_name ORDER BY unplanned_pct DESC;"),
    ("Q8: Unplanned Absence by Performance Rating", "SELECT p.rating, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct FROM fact_attendance a JOIN fact_performance p ON a.emp_id = p.emp_id WHERE a.work_date LIKE '2025%' AND p.review_year = 2024 GROUP BY p.rating ORDER BY p.rating;"),
    ("Q9: Employees Paid Above Job Band Max", "SELECT COUNT(*) as overpaid_count FROM dim_employee e JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND s.salary > j.max_salary;"),
    ("Q10: Average Tenure of Leavers by Reason", "SELECT termination_reason, ROUND(AVG((julianday(termination_date) - julianday(hire_date)) / 365.25), 1) as avg_tenure_years FROM dim_employee WHERE termination_date IS NOT NULL GROUP BY termination_reason;")
]

for title, q in queries:
    cursor.execute(q)
    print(f"\n--- {title} ---")
    for row in cursor.fetchall():
        print(row)

conn.close()
print("\n--- Project Completed Successfully! ---")

--- Step 1: Extract, Clean, and Load (ETL) ---


/tmp/ipykernel_2332/2246411189.py:30: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')


--- Step 2: Running All 10 Analytical Queries ---

--- Q1: Headcount & Avg Salary per Dept ---
('Engineering', 62, 3391.77)
('Sales', 49, 2888.98)
('Finance', 36, 2766.11)
('Marketing', 25, 2707.6)
('Human Resources', 30, 2472.67)
('Customer Support', 25, 2074.4)

--- Q2: Yearly Attrition (Leavers) ---
('2018', 1)
('2019', 12)
('2020', 24)
('2021', 11)
('2022', 18)
('2023', 25)
('2024', 29)
('2025', 24)
('2026', 29)

--- Q3: Top Leaver Depts & Reasons ---
('Customer Support', 'Better Offer', 24)
('Customer Support', 'Layoff', 14)
('Sales', 'Better Offer', 14)
('Sales', 'Layoff', 13)
('Customer Support', 'Relocation', 9)

--- Q4: Gender Pay Gap by Job Title ---
('Associate', 'Female', 2355.17)
('Associate', 'Male', 2402.35)
('Junior Associate', 'Female', 1704.62)
('Junior Associate', 'Male', 1697.05)
('Manager', 'Female', 6734.0)
('Manager', 'Male', 7275.71)
('Senior Associate', 'Female', 3480.45)
('Senior Associate', 'Male', 3611.11)
('Team Lead', 'Female', 4534.62)
('Team Lead', 'Male

In [65]:
import pandas as pd
import sqlite3

print("--- Loading and cleaning source files ---")

departments = pd.read_csv('departments.csv')
jobs = pd.read_csv('jobs.csv')
employees = pd.read_csv('employees.csv')
salaries = pd.read_csv('salaries.csv')
performance = pd.read_csv('performance_reviews.csv')
attendance = pd.read_csv('attendance.csv')

employees['first_name'] = employees['first_name'].str.strip()
employees['last_name'] = employees['last_name'].str.strip()
employees['email'] = employees['email'].str.lower()

def clean_gender(g):
    if pd.isna(g):
        return 'Unknown'
    g_str = str(g).strip().lower()
    if g_str in ['m', 'male']:
        return 'Male'
    elif g_str in ['f', 'female']:
        return 'Female'
    return g_str

employees['gender'] = employees['gender'].apply(clean_gender)
employees['birth_date'] = pd.to_datetime(employees['birth_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['hire_date'] = pd.to_datetime(employees['hire_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_reason'] = employees['termination_reason'].fillna('Active')

attendance = attendance.drop_duplicates()
attendance['is_unplanned_absence'] = attendance['status'].apply(lambda x: 1 if x in ['Sick', 'Absent'] else 0)

conn = sqlite3.connect('hr_warehouse.db')
departments.to_sql('dim_department', conn, if_exists='replace', index=False)
jobs.to_sql('dim_job', conn, if_exists='replace', index=False)
employees.to_sql('dim_employee', conn, if_exists='replace', index=False)
salaries.to_sql('fact_salary', conn, if_exists='replace', index=False)
performance.to_sql('fact_performance', conn, if_exists='replace', index=False)
attendance.to_sql('fact_attendance', conn, if_exists='replace', index=False)
conn.commit()
conn.close()

print("--- Database 'hr_warehouse.db' created and populated successfully! ---")

--- Loading and cleaning source files ---


/tmp/ipykernel_2332/3247368397.py:30: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')


--- Database 'hr_warehouse.db' created and populated successfully! ---


In [66]:
departments = pd.read_csv('departments.csv')
jobs = pd.read_csv('jobs.csv')
employees = pd.read_csv('employees.csv')
salaries = pd.read_csv('salaries.csv')
performance = pd.read_csv('performance_reviews.csv')
attendance = pd.read_csv('attendance.csv')

In [67]:
employees['first_name'] = employees['first_name'].str.strip()
employees['last_name'] = employees['last_name'].str.strip()
employees['email'] = employees['email'].str.lower()

In [68]:
def clean_gender(g):
    if pd.isna(g):
        return 'Unknown'
    g_str = str(g).strip().lower()
    if g_str in ['m', 'male']:
        return 'Male'
    elif g_str in ['f', 'female']:
        return 'Female'
    return g_str

employees['gender'] = employees['gender'].apply(clean_gender)

In [69]:
employees['birth_date'] = pd.to_datetime(employees['birth_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['hire_date'] = pd.to_datetime(employees['hire_date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')
employees['termination_reason'] = employees['termination_reason'].fillna('Active')

/tmp/ipykernel_2332/1340250398.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce').dt.strftime('%Y-%m-%d')


In [70]:
attendance = attendance.drop_duplicates()
attendance['is_unplanned_absence'] = attendance['status'].apply(lambda x: 1 if x in ['Sick', 'Absent'] else 0)

In [71]:
conn = sqlite3.connect('hr_warehouse.db')
departments.to_sql('dim_department', conn, if_exists='replace', index=False)
jobs.to_sql('dim_job', conn, if_exists='replace', index=False)
employees.to_sql('dim_employee', conn, if_exists='replace', index=False)
salaries.to_sql('fact_salary', conn, if_exists='replace', index=False)
performance.to_sql('fact_performance', conn, if_exists='replace', index=False)
attendance.to_sql('fact_attendance', conn, if_exists='replace', index=False)
conn.commit()
conn.close()

In [72]:
import sqlite3

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

queries = [
    ("Q1", "SELECT d.dept_name, COUNT(e.emp_id) as headcount, ROUND(AVG(s.salary), 2) as avg_salary FROM dim_department d JOIN dim_employee e ON d.dept_id = e.dept_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30' GROUP BY d.dept_name ORDER BY avg_salary DESC;"),
    ("Q2", "SELECT SUBSTR(termination_date, 1, 4) as year, COUNT(*) as leavers FROM dim_employee WHERE termination_date IS NOT NULL GROUP BY year ORDER BY year;"),
    ("Q3", "SELECT d.dept_name, e.termination_reason, COUNT(*) as cnt FROM dim_employee e JOIN dim_department d ON e.dept_id = d.dept_id WHERE e.termination_reason != 'Active' GROUP BY d.dept_name, e.termination_reason ORDER BY cnt DESC LIMIT 5;"),
    ("Q4", "SELECT j.job_title, e.gender, ROUND(AVG(s.salary), 2) as avg_sal FROM dim_employee e JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE s.to_date IS NULL OR s.to_date >= '2026-06-30' GROUP BY j.job_title, e.gender;"),
    ("Q5", "SELECT d.dept_name, j.job_title, s.salary FROM dim_employee e JOIN dim_department d ON e.dept_id = d.dept_id JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND j.job_title NOT LIKE '%Manager%' GROUP BY d.dept_name ORDER BY s.salary DESC;"),
    ("Q6", "SELECT p.rating, COUNT(*) as cnt FROM fact_performance p GROUP BY p.rating;"),
    ("Q7", "SELECT d.dept_name, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct FROM fact_attendance a JOIN dim_employee e ON a.emp_id = e.emp_id JOIN dim_department d ON e.dept_id = d.dept_id WHERE a.work_date LIKE '2025%' GROUP BY d.dept_name ORDER BY unplanned_pct DESC;"),
    ("Q8", "SELECT p.rating, ROUND(AVG(a.is_unplanned_absence) * 100, 2) as unplanned_pct FROM fact_attendance a JOIN fact_performance p ON a.emp_id = p.emp_id WHERE a.work_date LIKE '2025%' AND p.review_year = 2024 GROUP BY p.rating ORDER BY p.rating;"),
    ("Q9", "SELECT COUNT(*) as overpaid_count FROM dim_employee e JOIN dim_job j ON e.job_id = j.job_id JOIN fact_salary s ON e.emp_id = s.emp_id WHERE (s.to_date IS NULL OR s.to_date >= '2026-06-30') AND s.salary > j.max_salary;"),
    ("Q10", "SELECT termination_reason, ROUND(AVG((julianday(termination_date) - julianday(hire_date)) / 365.25), 1) as avg_tenure_years FROM dim_employee WHERE termination_date IS NOT NULL GROUP BY termination_reason;")
]

for name, q in queries:
    cursor.execute(q)
    print(f"--- {name} Results ---")
    for row in cursor.fetchall():
        print(row)

conn.close()

--- Q1 Results ---
('Engineering', 62, 3391.77)
('Sales', 49, 2888.98)
('Finance', 36, 2766.11)
('Marketing', 25, 2707.6)
('Human Resources', 30, 2472.67)
('Customer Support', 25, 2074.4)
--- Q2 Results ---
('2018', 1)
('2019', 12)
('2020', 24)
('2021', 11)
('2022', 18)
('2023', 25)
('2024', 29)
('2025', 24)
('2026', 29)
--- Q3 Results ---
('Customer Support', 'Better Offer', 24)
('Customer Support', 'Layoff', 14)
('Sales', 'Better Offer', 14)
('Sales', 'Layoff', 13)
('Customer Support', 'Relocation', 9)
--- Q4 Results ---
('Associate', 'Female', 2355.17)
('Associate', 'Male', 2402.35)
('Junior Associate', 'Female', 1704.62)
('Junior Associate', 'Male', 1697.05)
('Manager', 'Female', 6734.0)
('Manager', 'Male', 7275.71)
('Senior Associate', 'Female', 3480.45)
('Senior Associate', 'Male', 3611.11)
('Team Lead', 'Female', 4534.62)
('Team Lead', 'Male', 5201.43)
--- Q5 Results ---
('Engineering', 'Junior Associate', 1850.0)
('Marketing', 'Junior Associate', 1720.0)
('Sales', 'Junior Assoc

Q11

In [73]:
import sqlite3

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

q11_query = """
SELECT
    d.dept_name,
    ROUND(AVG((julianday('2026-06-30') - julianday(e.hire_date)) / 365.25), 1) as avg_tenure_years
FROM dim_employee e
JOIN dim_department d ON e.dept_id = d.dept_id
WHERE e.termination_date IS NULL
GROUP BY d.dept_name
ORDER BY avg_tenure_years DESC;
"""

cursor.execute(q11_query)
print("--- Q11: Avg Tenure of Active Employees per Dept ---")
for row in cursor.fetchall():
    print(row)

conn.close()

--- Q11: Avg Tenure of Active Employees per Dept ---
('Engineering', 4.0)
('Human Resources', 3.9)
('Sales', 3.7)
('Marketing', 3.2)
('Finance', 3.2)
('Customer Support', 2.9)


Q12

In [74]:
import sqlite3

conn = sqlite3.connect('hr_warehouse.db')
cursor = conn.cursor()

q12_query = """
SELECT
    d.dept_name,
    ROUND(SUM(CASE WHEN p.rating >= 4 THEN 1 ELSE 0 END) * 100.0 / COUNT(p.emp_id), 2) as high_performers_pct
FROM fact_performance p
JOIN dim_employee e ON p.emp_id = e.emp_id
JOIN dim_department d ON e.dept_id = d.dept_id
WHERE p.review_year = 2024
GROUP BY d.dept_name
ORDER BY high_performers_pct DESC;
"""

cursor.execute(q12_query)
print("--- Q12: High Performers Percentage (Rating 4-5) per Dept ---")
for row in cursor.fetchall():
    print(row)

conn.close()

--- Q12: High Performers Percentage (Rating 4-5) per Dept ---
('Sales', 38.46)
('Marketing', 36.36)
('Finance', 35.71)
('Engineering', 31.48)
('Human Resources', 30.43)
('Customer Support', 19.23)
